In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

import scripts.stock_plots as stock_plots
from scripts.preparation import download_data

In [2]:
def extract_ticker(df_row, russell_list):
    capital = 0
    for letter in df_row:
        if letter.isupper():
            capital += 1
        else:
            break
    
    answer = df_row[:capital-1]
    debug = answer

    while len(answer) > 0:
        if answer in russell_list:
            return answer
        else:
            answer = answer[:len(answer)-1]
    
    print("Can't find: ", debug)
    return None

In [3]:
large_cap = pd.read_excel("large_cap_option.xlsx")
ticker_list = large_cap.ticker.values
ticker_object = download_data(ticker_list)


In [4]:
# russell_table = pd.read_html("https://en.wikipedia.org/wiki/Russell_1000_Index")
# russell_list = list(russell_table[2]["Ticker"])

# top100 = pd.read_html("https://www.tradingview.com/markets/stocks-usa/market-movers-large-cap/")
# ticker_list = list(top100[0]["Symbol"].apply(extract_ticker, russell_list=russell_list))
# ticker_list = [i for i in ticker_list if i is not None]

# # ticker_list = ticker_list[:30]

# ticker_object = download_data(ticker_list)


# Start

In [5]:
count = 0
mom_score = []
candles = {}
momentums = {}
for ticker in ticker_list:
    stock_plot = stock_plots.PlotInfo(ticker_object.tickers[ticker], ticker, "80d")
    if np.all(stock_plot.df[["5ma", "10ma", "12ema", "20ema"]].diff().iloc[-3:] > 0):
    # if ((stock_plot.df["10ma"] > stock_plot.df["20ma"])[-3:].all() and
    #     (stock_plot.df["20ma"] > stock_plot.df["60ema"])[-3:].all() and
    #     (np.all(stock_plot.df[["10ma","20ema"]].diff().iloc[-3:] > 0))):
        # # Fair option value calculation, but not working due to delayed quote
        # op_expire_date = ticker_object.tickers[ticker].options[3]
        # call_op = ticker_object.tickers[ticker].option_chain(op_expire_date).calls
        # itm_idx = sum(call_op["inTheMoney"] == True) - 1
        # contract_value = call_op.iloc[itm_idx+1]["strike"] - call_op.iloc[itm_idx-1]["strike"]
        # contract_price = call_op.iloc[itm_idx-1][["bid", "ask"]].mean() - call_op.iloc[itm_idx+1][["bid", "ask"]].mean()
        # risk_reward = (contract_value - contract_price) / contract_price
        # print(ticker, risk_reward)
        # continue

        candle = stock_plot.generate_candle_plot_no_op()

        momentum = stock_plot.df["5ma"].subtract(stock_plot.df["5ma"].shift(3))/stock_plot.df["5ma"]
        momentum = np.round(momentum[-10:] * 100, 2)

        mom_score.append([np.mean(momentum[-5:]), ticker])
        candles[ticker] = candle
        momentums[ticker] = momentum

        count += 1

        break

mom_score.sort(reverse=True)

print(count)

1


In [11]:
stock_plot.df["J"]

Date
2024-03-05 00:00:00-05:00   NaN
2024-03-06 00:00:00-05:00   NaN
2024-03-07 00:00:00-05:00   NaN
2024-03-08 00:00:00-05:00   NaN
2024-03-11 00:00:00-04:00   NaN
                             ..
2024-06-21 00:00:00-04:00   NaN
2024-06-24 00:00:00-04:00   NaN
2024-06-25 00:00:00-04:00   NaN
2024-06-26 00:00:00-04:00   NaN
2024-06-27 00:00:00-04:00   NaN
Name: J, Length: 80, dtype: float64

In [6]:
for _, ticker in mom_score:
    print(momentums[ticker].values)
    candles[ticker].show()

[ 2.17  1.42  1.05  0.35 -0.38 -0.91 -0.83 -0.4   0.49  1.37]


# End